# Mini Research #3: Network Topology and Information Spread

## Research Question
**How does network topology affect the spread of information in a simulated social network?**

This project investigates the effect of network structure on information spreading using controlled computational simulations.

## 1. Background and Motivation

Information in a social network does not spread only because of the probability that one person passes information to another. The structure of the connections also determines how easily information can move through the network. Different network topologies create different patterns of connectivity, clustering, and highly connected nodes.

In this study, three synthetic network structures are compared: Random, Small-World, and Scale-Free networks. The goal is not to model real human behavior exactly, but to test whether the structure of a simulated network changes the speed and final extent of information spread under the same simulation assumptions.

## 2. Problem Statement

Two networks can contain the same number of simulated individuals while having very different connection patterns. Those structural differences may change how quickly information reaches a large fraction of the network.

The experiment therefore asks whether network topology produces measurable differences in information-spread dynamics when the number of nodes, transmission probability, number of simulation steps, and number of runs are controlled.

## 3. Objective and Hypothesis

**Objective:** Compare information spreading across Random, Small-World, and Scale-Free network topologies under the same simulation conditions.

**Hypothesis:** Network topology will affect the speed at which information spreads through the simulated network, even if the eventual final spread becomes similar across the topologies.

## 4. Experimental Design

The experiment uses 100 simulated nodes. One node is selected as the initial informed node, and information can be transmitted across network connections with a probability of 0.20 during each simulation step.

Each topology is simulated for 30 steps and repeated 30 times. A fixed random seed is used to make the experiment reproducible. The main evaluation measures are final spread after 30 steps and the time required to reach 50% informed nodes.

## 5. Network Topologies

### Random Network
An Erdős–Rényi random graph is used. Connections are generated probabilistically, producing a comparatively irregular network structure.

### Small-World Network
A Watts–Strogatz network is used. This model combines local clustering with relatively short paths between parts of the network.

### Scale-Free Network
A Barabási–Albert network is used. This model produces heterogeneous connectivity, including nodes with substantially more connections than many others.

## 6. Libraries and Parameters

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('Libraries imported successfully.')

In [ ]:
N_NODES = 100
TRANSMISSION_PROBABILITY = 0.20
SIMULATION_STEPS = 30
N_RUNS = 30
TARGET_SPREAD = 0.50
SEED = 42

print(f'Number of nodes: {N_NODES}')
print(f'Transmission probability: {TRANSMISSION_PROBABILITY}')
print(f'Simulation steps: {SIMULATION_STEPS}')
print(f'Number of runs: {N_RUNS}')
print(f'Target spread: {TARGET_SPREAD:.0%}')
print(f'Random seed: {SEED}')

## 7. Constructing the Networks

In [ ]:
random_network = nx.erdos_renyi_graph(n=N_NODES, p=0.05, seed=SEED)
small_world_network = nx.watts_strogatz_graph(n=N_NODES, k=6, p=0.1, seed=SEED)
scale_free_network = nx.barabasi_albert_graph(n=N_NODES, m=3, seed=SEED)

networks = {
    'Random': random_network,
    'Small-World': small_world_network,
    'Scale-Free': scale_free_network
}

for name, graph in networks.items():
    print(f'{name}: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges')

## 8. Network Structural Metrics

These descriptive metrics help characterize the three generated graphs before running the spread simulation.

In [ ]:
for name, graph in networks.items():
    avg_degree = np.mean([degree for _, degree in graph.degree()])
    avg_clustering = nx.average_clustering(graph)
    print(f'{name}:')
    print(f'  Average degree: {avg_degree:.2f}')
    print(f'  Average clustering: {avg_clustering:.4f}')


## 9. Visualizing Network Structure

In [ ]:
plt.figure(figsize=(15, 5))

for i, (name, graph) in enumerate(networks.items(), start=1):
    plt.subplot(1, 3, i)
    pos = nx.spring_layout(graph, seed=SEED)
    nx.draw_networkx(graph, pos=pos, node_size=25, with_labels=False, width=0.5)
    plt.title(name)
    plt.axis('off')

plt.suptitle('Simulated Network Topologies')
plt.tight_layout()
plt.show()

## 10. Information-Spread Simulation

The simulation starts with one randomly selected informed node. During each step, every informed node attempts to transmit information to each uninformed neighbor. A transmission succeeds with probability 0.20. Newly informed nodes participate in later steps.

In [ ]:
def simulate_spread(graph, transmission_probability, steps, rng):
    nodes = list(graph.nodes())
    initial_node = rng.choice(nodes)
    informed = {initial_node}
    spread_over_time = [len(informed)]

    for step in range(steps):
        new_informed = set(informed)
        for node in informed:
            for neighbor in graph.neighbors(node):
                if neighbor not in informed:
                    if rng.random() < transmission_probability:
                        new_informed.add(neighbor)
        informed = new_informed
        spread_over_time.append(len(informed))

    return spread_over_time

## 11. Running the Experiment

Each topology is simulated 30 times. Every run records the number and fraction of informed nodes at every step. This produces 3 × 30 × 31 = 2,790 observations.

In [ ]:
all_results = []

for network_name, graph in networks.items():
    for run in range(N_RUNS):
        rng = np.random.default_rng(SEED + run)
        spread = simulate_spread(
            graph=graph,
            transmission_probability=TRANSMISSION_PROBABILITY,
            steps=SIMULATION_STEPS,
            rng=rng
        )

        for step, informed_count in enumerate(spread):
            all_results.append({
                'Network': network_name,
                'Run': run + 1,
                'Step': step,
                'Informed': informed_count,
                'Spread': informed_count / N_NODES
            })

results_df = pd.DataFrame(all_results)

print('Simulation completed.')
print(f'Total records: {len(results_df)}')
print(f'Networks: {results_df["Network"].nunique()}')
print(f'Runs per network: {results_df["Run"].nunique()}')

## 12. Calculating Evaluation Metrics

In [ ]:
summary_results = []

for network_name in networks.keys():
    network_data = results_df[results_df['Network'] == network_name]

    final_spread_per_run = (
        network_data[network_data['Step'] == SIMULATION_STEPS]
        .groupby('Run')['Spread']
        .first()
    )

    mean_final_spread = final_spread_per_run.mean()

    times_to_target = []
    for run in sorted(network_data['Run'].unique()):
        run_data = network_data[network_data['Run'] == run]
        reached = run_data[run_data['Spread'] >= TARGET_SPREAD]
        if len(reached) > 0:
            time_to_target = reached['Step'].iloc[0]
        else:
            time_to_target = np.nan
        times_to_target.append(time_to_target)

    mean_time_to_target = np.nanmean(times_to_target)

    summary_results.append({
        'Network': network_name,
        'Final Spread': mean_final_spread,
        'Time to 50%': mean_time_to_target
    })

summary_df = pd.DataFrame(summary_results)
summary_df

## 13. Mean Information Spread Over Time

This figure is generated directly from the simulation output. It shows how the average fraction of informed nodes changes across the 30 simulation steps.

In [ ]:
spread_over_time = (
    results_df
    .groupby(['Network', 'Step'])['Spread']
    .mean()
    .reset_index()
)

plt.figure(figsize=(10, 6))

for network_name in networks.keys():
    network_data = spread_over_time[
        spread_over_time['Network'] == network_name
    ]
    plt.plot(
        network_data['Step'],
        network_data['Spread'] * 100,
        label=network_name,
        linewidth=2.5
    )

plt.axhline(y=50, linestyle='--', linewidth=1.5, label='50% Spread')
plt.xlabel('Simulation Step')
plt.ylabel('Informed Nodes (%)')
plt.title('Information Spread Across Different Network Topologies')
plt.ylim(0, 105)
plt.grid(True, alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig('spread_over_time.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: spread_over_time.png')

## 14. Time to Reach 50% Spread

In [ ]:
final_results = summary_df.copy()
final_results['Final Spread (%)'] = final_results['Final Spread'] * 100
final_results = final_results[['Network', 'Final Spread (%)', 'Time to 50%']]

plt.figure(figsize=(8, 5))
bars = plt.bar(
    final_results['Network'],
    final_results['Time to 50%'],
    width=0.6
)
plt.ylabel('Simulation Steps')
plt.xlabel('Network Topology')
plt.title('Time to Reach 50% Information Spread')
plt.grid(axis='y', alpha=0.25)

for bar, value in zip(bars, final_results['Time to 50%']):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.15,
        f'{value:.2f}',
        ha='center', va='bottom', fontsize=10
    )

plt.tight_layout()
plt.savefig('time_to_50.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: time_to_50.png')

## 15. Final Spread After 30 Steps

In [ ]:
plt.figure(figsize=(8, 5))
bars = plt.bar(
    final_results['Network'],
    final_results['Final Spread (%)'],
    width=0.6
)
plt.ylabel('Final Informed Nodes (%)')
plt.xlabel('Network Topology')
plt.title('Final Information Spread After 30 Steps')
plt.ylim(0, 105)
plt.grid(axis='y', alpha=0.25)

for bar, value in zip(bars, final_results['Final Spread (%)']):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1,
        f'{value:.2f}%',
        ha='center', va='bottom', fontsize=10
    )

plt.tight_layout()
plt.savefig('final_spread.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: final_spread.png')

## 16. Exporting Final Results

In [ ]:
final_results.to_csv('final_results.csv', index=False)

print('Final results exported successfully.')
print('\nFinal Results:')
print(final_results.to_string(index=False))

## 17. Validation Checks

These checks confirm that the generated dataset has the expected structure and that spread values remain valid.

In [ ]:
expected_records = 3 * N_RUNS * (SIMULATION_STEPS + 1)

checks = {
    'Expected record count': len(results_df) == expected_records,
    'Spread values within [0, 1]': results_df['Spread'].between(0, 1).all(),
    'All graphs have 100 nodes': all(g.number_of_nodes() == N_NODES for g in networks.values()),
    'Three network types present': set(results_df['Network']) == set(networks.keys())
}

for check, passed in checks.items():
    print(f'{check}: {passed}')

print(f'Expected records: {expected_records}')

## 18. Results and Interpretation

Under the simulated conditions, the Scale-Free topology reached 50% information spread fastest, with an average time of 5.93 simulation steps. The Random network required 9.00 steps, while the Small-World network required 9.20 steps.

By the end of 30 steps, all three topologies reached approximately complete spread: Random 99.93%, Small-World 100.00%, and Scale-Free 100.00%.

The main observed difference was therefore the **speed of spreading**, rather than the final percentage reached within the 30-step simulation window.

## 19. Limitations

This experiment is a synthetic computational simulation and should not be interpreted as a direct measurement of real human rumor or information behavior. The transmission probability of 0.20 is a modeling assumption, not an empirical estimate.

The network models also do not have identical edge counts: the generated Random network contains 224 edges, the Small-World network 300 edges, and the Scale-Free network 291 edges. Therefore, the experiment compares standard topology models under the selected generation parameters rather than isolating topology while holding every structural property identical.

Results can also depend on the chosen number of nodes, transmission probability, number of steps, and random seed.

## 20. Reproducibility and Future Work

The experiment uses fixed parameters and a fixed seed so that the computational procedure can be reproduced. A useful extension would be to repeat the experiment across multiple transmission probabilities, network sizes, and topology-generation parameters. Another extension would be to normalize structural properties such as edge count before comparing topologies.

## Conclusion

Under the simulated conditions, network topology affected the speed of information spread. The Scale-Free network reached the 50% target substantially earlier than the Random and Small-World networks, while all three networks approached complete spread by the end of the 30-step simulation. These findings support the hypothesis for this simulation setup, while the limitations prevent direct generalization to real-world social networks.